<br/>
<img src="images/utfsm.png" alt="" width="130px" align="left"/>
<img src="images/utfsm.png" alt="" width="130px" align="right"/>
<div align="center">
<h1>Redes Profundas y Arquitecturas Residuales ResNet</h1>
<br/><br/>
Dr. Nicolás Gálvez Ramírez<br/>
Dr. Patricio Olivares Roncagliolo<br/><br/>
Ingeniería Civil Telemática<br/>
Departamento de Eléctronica<br/>
Universidad Técnica Federico Santa María
</div>
<br>
Fuentes: 
<br>
"Hands-on Machine Learning with Scikit-Learn, Keras & TensorFlow"

## 🎯 Objetivos de Aprendizaje

Al finalizar esta clase, el estudiante será capaz de:

1. **Comprender** las innovaciones de VGG, ResNet, Inception y MobileNet.
2. **Explicar** las conexiones residuales y su papel frente al *vanishing gradient*.
3. **Aplicar** *transfer learning* con modelos preentrenados en ImageNet (Keras Applications).
4. **Realizar** *fine-tuning* de una arquitectura para un dominio específico.
5. **Comparar** los *trade-offs* entre accuracy, número de parámetros y latencia.

> 💡 **Prerrequisitos:** [04_RedesNeuronales](../../04_RedesNeuronales/04_RedesNeuronales.ipynb).

# Redes Neuronales, CNNs y Motivación para Redes Más Profundas

- Las **redes neuronales (NN)** permiten resolver tareas como clasificación de imágenes y reconocimiento de patrones.
- Las **redes convolucionales (CNNs)** son NN especializadas para datos con estructura espacial, como imágenes.
- Los modelos que hemos usado funcionan bien para problemas simples.
- Para problemas más complejos, se necesita mayor capacidad de representación.
- Una forma de lograrlo es **agregar más capas**, para aprender características más abstractas.

## Ejemplo: Comparación dos redes CNN con dataset CIFAR-10


In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

from tensorflow.keras.datasets import cifar10
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping


# Cargar datos CIFAR-10
(x_train, y_train), (x_test, y_test) = cifar10.load_data()
x_train, x_test = x_train / 255.0, x_test / 255.0
y_train, y_test = to_categorical(y_train, 10), to_categorical(y_test, 10)


# Modelo sencillo
simple_model = Sequential([
    Conv2D(32, (3,3), activation='relu', input_shape=(32,32,3)),
    MaxPooling2D(2,2),
    Flatten(),
    Dense(32, activation='relu'),
    Dense(10, activation='softmax')
])
simple_model.compile(optimizer='adam',
                     loss='categorical_crossentropy',
                     metrics=['accuracy'])

# Modelo más profundo
deep_model = Sequential([
    Conv2D(32, (3,3), activation='relu', input_shape=(32,32,3)),
    Conv2D(32, (3,3), activation='relu'),
    MaxPooling2D(2,2),
    Conv2D(64, (3,3), activation='relu'),
    Conv2D(64, (3,3), activation='relu'),
    MaxPooling2D(2,2),
    Flatten(),
    Dense(64, activation='relu'),
    Dense(10, activation='softmax')
])
deep_model.compile(optimizer='adam',
                   loss='categorical_crossentropy',
                   metrics=['accuracy'])

# Definir EarlyStopping
early_stop = EarlyStopping(monitor='val_loss',
                           patience=5, 
                           restore_best_weights=True)


In [ ]:
# Entrenamiento corto para comparar
simple_history = simple_model.fit(x_train, y_train, epochs=100, batch_size=64, validation_data=(x_test, y_test), callbacks=[early_stop])

In [ ]:
deep_history = deep_model.fit(x_train, y_train, epochs=100, batch_size=64, validation_data=(x_test, y_test), callbacks=[early_stop])

In [ ]:
print(f"Precisión modelo simple: {simple_history.history['val_accuracy'][-1]:.4f}")
print(f"Precisión modelo más profundo: {deep_history.history['val_accuracy'][-1]:.4f}")

- Este ejemplo muestra que una red más profunda puede lograr mejores resultados en el mismo problema.
- Sin embargo, aumentar la profundidad también trae nuevos retos.

# El Desafío de la Profundidad

- Aumentar la cantidad de capas no siempre mejora el rendimiento.
- Redes demasiado profundas pueden tener **peor precisión** que redes más pequeñas.
- Este problema se conoce como **problema de degradación**.
- No es lo mismo que el sobreajuste: la red profunda puede tener mayor error incluso en entrenamiento.
- Surgen dificultades como la desaparición o explosión de gradientes al retropropagar el error.
- Estos problemas motivaron nuevas arquitecturas para entrenar redes profundas de forma estable.


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense
from tensorflow.keras.callbacks import EarlyStopping

# Red CNN sobrecompleja para CIFAR-10
overdeep_model = Sequential()
overdeep_model.add(Conv2D(32, (3,3), activation='relu', input_shape=(32,32,3)))

# Añadir muchas capas sin skip connections
for _ in range(10):
    overdeep_model.add(Conv2D(32, (3,3), activation='relu', padding='same'))

overdeep_model.add(MaxPooling2D(2,2))
overdeep_model.add(Flatten())
overdeep_model.add(Dense(64, activation='relu'))
overdeep_model.add(Dense(10, activation='softmax'))

overdeep_model.compile(optimizer='adam',
                       loss='categorical_crossentropy',
                       metrics=['accuracy'])

# Definir EarlyStopping
early_stop = EarlyStopping(monitor='val_loss',
                           patience=5,  
                           restore_best_weights=True)


# Entrenar misma cantidad de épocas para comparar
overdeep_history = overdeep_model.fit(x_train, y_train,
                                      epochs=100,
                                      batch_size=64,
                                      validation_data=(x_test, y_test),
                                      callbacks=[early_stop])

print(f"Precisión modelo sobrecomplejo: {overdeep_history.history['val_accuracy'][-1]:.4f}")


In [ ]:
print(f"Precisión modelo simple: {simple_history.history['val_accuracy'][-1]:.4f}")
print(f"Precisión modelo más profundo: {deep_history.history['val_accuracy'][-1]:.4f}")
print(f"Precisión modelo sobrecomplejo: {overdeep_history.history['val_accuracy'][-1]:.4f}")

- Esta red, mucho más profunda, puede terminar rindiendo peor que la red más pequeña.
- El rendimiento más bajo en entrenamiento muestra el **problema de degradación**.
- Para resolver esto se necesitan nuevas **formas de arquitectura**.

## Vanishing y Exploding Gradients

- En redes profundas, los gradientes se propagan hacia atrás para actualizar los pesos.
- Si los gradientes se vuelven muy pequeños (**vanishing**), las capas iniciales casi no aprenden.
- Si los gradientes se vuelven muy grandes (**exploding**), el entrenamiento se vuelve inestable.
- Estos problemas dificultan entrenar redes profundas sin técnicas adicionales.
- El ejemplo de la CNN sobrecompleja entrenada antes muestra cómo estos problemas afectan el rendimiento.
- La normalización de lotes ayuda, pero no siempre es suficiente.
- En la siguiente parte veremos cómo se usan **conexiones residuales** para mejorar esto.


# ResNet y Conexiones Residuales

- Para resolver los problemas de degradación y gradientes, se propuso la arquitectura **ResNet**.
- ResNet introduce las **conexiones residuales** o *skip connections*.
- Estas conexiones permiten saltar capas y suman directamente la entrada a la salida de un bloque.
- Así, la red puede aprender más fácilmente la función identidad si es necesario.
- Esto mejora el flujo de gradientes y facilita entrenar redes muy profundas.
- En la práctica, ResNet permite construir redes con cientos de capas sin perder rendimiento.


## Bloques residuales

<img src="images/resnet.png" alt="" width="400px"/>

- Combina la entrada original con la salida de un par de capas convolucionales.
- Incluye:
  - Dos `Conv2D` + `BatchNormalization` → transforman la entrada.
  - `Add()` → suma la entrada original (**skip connection**) para facilitar aprender la identidad.
  - `ReLU` final → mantiene la no linealidad.
- Justificación:
  - Ayuda a entrenar redes profundas sin degradación.
  - Mejora el flujo de gradientes.
  - Permite aprender ajustes sobre la entrada sin perder información.



In [ ]:
# Función que define un bloque residual básico
def residual_block(x, filters, stride=1, use_projection=False):
    shortcut = x

    # Ruta principal
    y = layers.Conv2D(filters, 3, strides=stride, padding='same')(x)
    y = layers.BatchNormalization()(y)
    y = layers.ReLU()(y)
    y = layers.Conv2D(filters, 3, padding='same')(y)
    y = layers.BatchNormalization()(y)

    # Proyección si cambia tamaño o filtros
    if use_projection:
        shortcut = layers.Conv2D(filters, 1, strides=stride, padding='same')(shortcut)
        shortcut = layers.BatchNormalization()(shortcut)

    out = layers.Add()([shortcut, y])
    out = layers.ReLU()(out)
    return out

def build_simple_resnet(input_shape, num_classes):
    inputs = keras.Input(shape=input_shape)

    # Stem
    x = layers.Conv2D(32, 3, strides=1, padding='same')(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)

    # Etapa 1
    x = residual_block(x, filters=32)
    x = residual_block(x, filters=32)

    # Etapa 2 con downsampling
    x = residual_block(x, filters=64, stride=2, use_projection=True)
    x = residual_block(x, filters=64)

    # Etapa 3 con downsampling
    x = residual_block(x, filters=128, stride=2, use_projection=True)
    x = residual_block(x, filters=128)

    # Clasificador
    x = layers.GlobalAveragePooling2D()(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)

    model = keras.Model(inputs, outputs, name="SimpleResNet")
    return model


In [ ]:
# Crear y compilar SimpleResNet
resnet_model = build_simple_resnet((32, 32, 3), 10)
resnet_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

# Callback EarlyStopping
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

# Hiperparámetros
batch_size = 64
epochs = 100

# Entrenar SimpleResNet
history_resnet = resnet_model.fit(
    x_train, y_train,
    batch_size=batch_size,
    epochs=epochs,
    validation_data=(x_test, y_test),
    callbacks=[early_stop]
)

In [ ]:
print(f"Precisión modelo SimpleResNet: {history_resnet.history['val_accuracy'][-1]:.4f}")

## Prueba de ResNet50 entrenada desde cero con CIFAR-10

- TensorFlow incluye arquitecturas como ResNet50 listas para usar.
- Aquí se carga ResNet50 sin pesos preentrenados para entrenarla desde cero.
- Se adapta la capa final para clasificar las 10 clases de CIFAR-10.
- Esto permite comparar su rendimiento con PlainNet y SimpleResNet.


In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import ResNet50
from tensorflow.keras import layers, models
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping

# Cargar CIFAR-10
(x_train, y_train), (x_test, y_test) = cifar10.load_data()
num_classes = 10

# Normalizar y codificar etiquetas
y_train = to_categorical(y_train, num_classes)
y_test = to_categorical(y_test, num_classes)

# Crear datasets tf.data con redimensionamiento
def preprocess(image, label):
    image = tf.cast(image, tf.float32) / 255.0
    image = tf.image.resize(image, (224, 224))
    return image, label

batch_size = 64

# Crea los conjuntos de datos de entrenamiento y validación usando la API tf.data:
# - from_tensor_slices: convierte los tensores (x, y) en un dataset
# - shuffle: mezcla los datos (solo en entrenamiento)
# - map: aplica la función de preprocesamiento en paralelo
# - batch: agrupa los datos en lotes del tamaño especificado
# - prefetch: prepara el siguiente lote mientras se entrena para mejorar el rendimiento

train_ds = (
    tf.data.Dataset.from_tensor_slices((x_train, y_train))
    .shuffle(buffer_size=5000)
    .map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(batch_size)
    .prefetch(tf.data.AUTOTUNE)
)

val_ds = (
    tf.data.Dataset.from_tensor_slices((x_test, y_test))
    .map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(batch_size)
    .prefetch(tf.data.AUTOTUNE)
)

# Cargar ResNet50 sin pesos preentrenados
base_model = ResNet50(weights=None, include_top=False, input_shape=(224, 224, 3))

# Construir modelo final
inputs = tf.keras.Input(shape=(224, 224, 3))
x = base_model(inputs)
x = layers.GlobalAveragePooling2D()(x)
outputs = layers.Dense(num_classes, activation='softmax')(x)

model = models.Model(inputs, outputs)

# Compilar
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# EarlyStopping
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

# Entrenar usando dataset eficiente
history_resnet50 = model.fit(
    train_ds,
    epochs=100,
    validation_data=val_ds,
    callbacks=[early_stop]
)

# Precisión final
print(f"Precisión ResNet50 desde cero: {history_resnet50.history['val_accuracy'][-1]:.4f}")


In [ ]:
print(f"Precisión modelo ResNet50: {history_resnet50.history['val_accuracy'][-1]:.4f}")

## ¿Qué demuestra ResNet?

- Simplemente apilar más capas no siempre mejora el rendimiento: puede aparecer el problema de degradación.
- Las **conexiones residuales** permiten que la red aprenda funciones identidad fácilmente y ajusten solo lo necesario.
- Esto ayuda a mantener el flujo de gradientes y facilita entrenar redes profundas de forma estable.
- ResNet abrió la puerta a arquitecturas con cientos de capas, que hoy se usan ampliamente en visión por computador y otros problemas complejos.
- En la práctica, se logra mejor precisión y convergencia más estable que con redes planas de profundidad comparable.


## Introducción a Inception

- **Inception** es una arquitectura de red neuronal profunda diseñada para procesar imágenes de forma más eficiente.
- Combina **múltiples tipos de filtros** (1x1, 3x3, 5x5) y pooling en paralelo dentro de un mismo bloque (módulo Inception).
- Cada módulo captura patrones a diferentes escalas y los concatena, mejorando la representación sin aumentar excesivamente el costo computacional.
- Fue popularizada por la familia **GoogLeNet** (Inception v1) y mejorada en versiones como Inception v2, v3 y v4.
- Se usa ampliamente en visión por computador para tareas de clasificación, detección y reconocimiento de imágenes.
- Las arquitecturas Inception destacan por su **eficiencia computacional** y su buen rendimiento con imágenes de alta resolución.


## ¿Qué hace un bloque Inception?

<img src="images/inception.png" alt="" width="600px"/>


- Un bloque Inception combina **varias ramas de filtros en paralelo**:
  - **¿Qué significa una rama?**  
    Cada rama es un camino independiente que aplica una secuencia de operaciones de convolución o pooling sobre la **misma entrada**.  
    Por ejemplo, una rama usa solo filtros 1x1, otra combina 1x1 y 3x3, otra usa 5x5, etc.
  - Cada rama extrae **características diferentes** de la misma entrada, con distintos tamaños de filtro y estrategias de agrupación.
  - Al final, todas las ramas se **concatenan** para formar la salida del bloque, combinando información de múltiples escalas.

- Ramas típicas en Inception v1:
  - **Rama 1:** Filtro 1x1 para extraer características básicas.
  - **Rama 2:** Filtro 1x1 seguido de 3x3 para detalles locales.
  - **Rama 3:** Filtro 1x1 seguido de 5x5 para patrones más amplios.
  - **Rama 4:** MaxPooling 3x3 seguido de 1x1 conv para robustez y reducción de dimensiones.

- Esta combinación permite capturar información a diferentes escalas dentro de un solo bloque, manteniendo la eficiencia computacional.
- Por eso los bloques Inception se apilan para construir arquitecturas profundas, como GoogLeNet (Inception v1) y sus versiones mejoradas.


In [ ]:
from tensorflow.keras import layers, Model, Input

def inception_block(x, f1x1, f3x3_reduce, f3x3, f5x5_reduce, f5x5, pool_proj):
    # Rama 1: 1x1 conv
    path1 = layers.Conv2D(f1x1, (1, 1), padding='same', activation='relu')(x)
    
    # Rama 2: 1x1 conv + 3x3 conv
    path2 = layers.Conv2D(f3x3_reduce, (1, 1), padding='same', activation='relu')(x)
    path2 = layers.Conv2D(f3x3, (3, 3), padding='same', activation='relu')(path2)
    
    # Rama 3: 1x1 conv + 5x5 conv
    path3 = layers.Conv2D(f5x5_reduce, (1, 1), padding='same', activation='relu')(x)
    path3 = layers.Conv2D(f5x5, (5, 5), padding='same', activation='relu')(path3)
    
    # Rama 4: 3x3 max pooling + 1x1 conv
    path4 = layers.MaxPooling2D((3, 3), strides=(1, 1), padding='same')(x)
    path4 = layers.Conv2D(pool_proj, (1, 1), padding='same', activation='relu')(path4)
    
    # Concatenar todas las ramas
    output = layers.concatenate([path1, path2, path3, path4], axis=-1)
    return output

# Ejemplo de uso del bloque
inputs = Input(shape=(32, 32, 192))
x = inception_block(inputs, 64, 96, 128, 16, 32, 32)
model = Model(inputs, x)
model.summary()

## Prueba de InceptionV3

- La arquitectura **Inception** usa bloques que combinan filtros de distintos tamaños para capturar patrones a varias escalas.
- InceptionV3 es una versión optimizada y muy popular para tareas de clasificación de imágenes.
- TensorFlow permite cargar InceptionV3 ya entrenada en ImageNet.
- En este ejemplo se muestra cómo usarla para hacer una predicción sobre una imagen de CIFAR-10, redimensionada para que coincida con la entrada requerida.
- Esta prueba permite entender cómo funcionan modelos grandes sin necesidad de entrenarlos desde cero.


In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import InceptionV3
from tensorflow.keras import layers, models
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping

# Cargar CIFAR-10
(x_train, y_train), (x_test, y_test) = cifar10.load_data()
num_classes = 10

# Crear tf.data.Dataset para redimensionar
def preprocess(image, label):
    image = tf.cast(image, tf.float32) / 255.0
    image = tf.image.resize(image, (299, 299))
    return image, tf.one_hot(label[0], num_classes)

batch_size = 64

train_ds = (
    tf.data.Dataset.from_tensor_slices((x_train, y_train))
    .shuffle(5000)
    .map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(batch_size)
    .prefetch(tf.data.AUTOTUNE)
)

val_ds = (
    tf.data.Dataset.from_tensor_slices((x_test, y_test))
    .map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(batch_size)
    .prefetch(tf.data.AUTOTUNE)
)

# Cargar InceptionV3 sin pesos y sin top layer
base_model = InceptionV3(weights=None, include_top=False, input_shape=(299, 299, 3))

# Agregar capa final para CIFAR-10
inputs = tf.keras.Input(shape=(299, 299, 3))
x = base_model(inputs)
x = layers.GlobalAveragePooling2D()(x)
outputs = layers.Dense(num_classes, activation='softmax')(x)

model = models.Model(inputs, outputs)

# Compilar
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# EarlyStopping
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

# Entrenar
history_inception = model.fit(
    train_ds,
    epochs=100,
    validation_data=val_ds,
    callbacks=[early_stop]
)

# Mostrar accuracy final
print(f"Precisión InceptionV3 desde cero: {history_inception.history['val_accuracy'][-1]:.4f}")



In [ ]:
# Mostrar accuracy final
print(f"Precisión InceptionV3 desde cero: {history_inception.history['val_accuracy'][-1]:.4f}")

## Resumen de Inception

- Inception muestra cómo una arquitectura bien diseñada puede capturar patrones a múltiples escalas dentro de un mismo módulo.
- Aunque es más compleja que una CNN tradicional, su uso eficiente de filtros paralelos la hace competitiva en problemas de clasificación de imágenes.


## Introducción a EfficientNet

- **EfficientNet** es una familia de arquitecturas de redes neuronales convolucionales optimizadas para obtener la mejor relación entre precisión y eficiencia computacional.
- La idea central es el **compound scaling**: escalar **profundidad**, **anchura** (número de filtros) y **resolución de entrada** de forma balanceada y sistemática.
- EfficientNet combina principios de arquitecturas previas como MobileNetV2 y la búsqueda automática de arquitecturas (AutoML).
- La unidad básica es el bloque **MBConv**, que combina:
  - Convolución expandida (expansion layer).
  - Depthwise Separable Convolution (más eficiente).
  - Squeeze-and-Excitation (SE Block) para recalibrar características.
- Gracias a esta estructura, EfficientNet logra alta precisión con menos parámetros que arquitecturas tradicionales como ResNet o Inception.


## ¿Qué es un bloque MBConv?

- **MBConv (Mobile Inverted Bottleneck Conv)** es el bloque principal de EfficientNet.
- Combina algunas ideas clave:
  - **Expansión:** Aumenta temporalmente el número de canales con una convolución 1x1.
  - **Depthwise Convolution:** Aplica filtros independientes a cada canal, reduciendo cálculos.
  - **Proyección:** Reduce de nuevo los canales con otra convolución 1x1.
  - **Squeeze-and-Excitation (SE Block):** Recalibra los canales para resaltar los más relevantes.
- El término *Inverted Bottleneck* se refiere a que, a diferencia de un bottleneck clásico (Reducir → Procesar → Expandir), aquí se hace *Expandir → Procesar → Reducir*.
- Este bloque mantiene la eficiencia mientras captura características complejas.

**¿Por qué es más eficiente?**  

Porque realiza la mayor parte del procesamiento en canales expandidos usando **convoluciones depthwise**, que requieren muchos menos parámetros y operaciones que las convoluciones estándar, logrando **más precisión con menor costo computacional**.

In [ ]:
from tensorflow.keras import layers, Model, Input

def squeeze_excite_block(inputs, ratio=4):
    # Bloque SE: recalibra la importancia de cada canal de características
    filters = inputs.shape[-1]
    se = layers.GlobalAveragePooling2D()(inputs)
    se = layers.Dense(filters // ratio, activation='relu')(se)
    se = layers.Dense(filters, activation='sigmoid')(se)
    se = layers.Reshape((1, 1, filters))(se)
    return layers.multiply([inputs, se])

def mbconv_block(x, expansion_factor=6, filters=16, stride=1):
    # Bloque MBConv: base de EfficientNet (expandir → procesar → reducir)
    in_channels = x.shape[-1]
    # Expandir
    x_expanded = layers.Conv2D(in_channels * expansion_factor, 1, padding='same', activation='relu')(x)
    x_expanded = layers.BatchNormalization()(x_expanded)

    # Depthwise separable convolution
    x_depthwise = layers.DepthwiseConv2D(3, strides=stride, padding='same', activation='relu')(x_expanded)
    x_depthwise = layers.BatchNormalization()(x_depthwise)

    # Squeeze-and-Excitation
    x_se = squeeze_excite_block(x_depthwise)

    # Proyección
    x_projected = layers.Conv2D(filters, 1, padding='same')(x_se)
    x_projected = layers.BatchNormalization()(x_projected)

    # Skip connection si no hay cambio de forma
    if stride == 1 and in_channels == filters:
        x_out = layers.Add()([x, x_projected])
    else:
        x_out = x_projected

    return x_out

# Ejemplo de uso: entrada y un bloque MBConv
inputs = Input(shape=(32, 32, 16))
x = mbconv_block(inputs, expansion_factor=6, filters=16, stride=1)
model = Model(inputs, x)
model.summary()


## Prueba de EfficientNetB0

- EfficientNetB0 es la versión base de la familia EfficientNet, diseñada para ofrecer un buen equilibrio entre precisión y eficiencia.
- TensorFlow permite cargar EfficientNetB0 con pesos preentrenados en ImageNet.
- En este ejemplo se muestra cómo usarla para predecir una clase en una imagen de CIFAR-10, redimensionada a la entrada esperada.
- Esta prueba permite explorar cómo funcionan las arquitecturas modernas de redes eficientes sin necesidad de entrenarlas desde cero.


In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras import layers, models
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping

# Cargar CIFAR-10
(x_train, y_train), (x_test, y_test) = cifar10.load_data()
num_classes = 10

# Dataset tf.data con resize dinámico
def preprocess(image, label):
    image = tf.cast(image, tf.float32) / 255.0
    image = tf.image.resize(image, (224, 224))
    return image, tf.one_hot(label[0], num_classes)

batch_size = 64

train_ds = (
    tf.data.Dataset.from_tensor_slices((x_train, y_train))
    .shuffle(5000)
    .map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(batch_size)
    .prefetch(tf.data.AUTOTUNE)
)

val_ds = (
    tf.data.Dataset.from_tensor_slices((x_test, y_test))
    .map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(batch_size)
    .prefetch(tf.data.AUTOTUNE)
)

# Cargar EfficientNetB0 SIN pesos preentrenados (o pon weights='imagenet' para transfer learning)
base_model = EfficientNetB0(weights=None, include_top=False, input_shape=(224, 224, 3))

# Construir modelo final para CIFAR-10
inputs = tf.keras.Input(shape=(224, 224, 3))
x = base_model(inputs)
x = layers.GlobalAveragePooling2D()(x)
outputs = layers.Dense(num_classes, activation='softmax')(x)

model = models.Model(inputs, outputs)

# Compilar
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# EarlyStopping
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

# Entrenar
history_efficientnet = model.fit(
    train_ds,
    epochs=100,
    validation_data=val_ds,
    callbacks=[early_stop]
)

In [ ]:
# Mostrar precisión final en validación
print(f"Precisión EfficientNetB0 desde cero: {history_efficientnet.history['val_accuracy'][-1]:.4f}")

# Técnicas de Optimización y Regularización en Redes Profundas

En esta parte revisaremos estrategias prácticas para mejorar el rendimiento y la generalización de redes neuronales convolucionales.  
Estas técnicas permiten:
- Acelerar y estabilizar el entrenamiento.
- Hacer que los modelos sean más robustos frente a nuevas muestras.
- Reducir el sobreajuste.

Veremos de forma resumida:
- **Batch Normalization** para estabilizar activaciones internas.
- **Data Augmentation Avanzada** con **Cutout** y **Mixup**.
- **Label Smoothing** para suavizar etiquetas

## 1. Batch Normalization

- Reduce el **Internal Covariate Shift**, estabilizando la distribución de activaciones internas.
- Normaliza cada mini-batch con media 0 y varianza 1.
- Incluye parámetros aprendibles (**$\gamma$**, **$\beta$**) para mantener flexibilidad.
- Beneficios: entrenamiento más rápido y estable, efecto regularizador.
- Normalmente se coloca después de `Conv2D` o `Dense` y antes de la activación.



In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping

# Dataset CIFAR-10
(x_train, y_train), (x_test, y_test) = keras.datasets.cifar10.load_data()
x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0

num_classes = 10
y_train_cat = keras.utils.to_categorical(y_train, num_classes)
y_test_cat = keras.utils.to_categorical(y_test, num_classes)

# Modelo SIN Batch Normalization
def build_model_no_bn(input_shape, num_classes):
    model = keras.Sequential([
        layers.Conv2D(32, (3, 3), padding='same', activation='relu', input_shape=input_shape),
        layers.Conv2D(32, (3, 3), padding='same', activation='relu'),
        layers.MaxPooling2D((2, 2)),
        
        layers.Conv2D(64, (3, 3), padding='same', activation='relu'),
        layers.Conv2D(64, (3, 3), padding='same', activation='relu'),
        layers.MaxPooling2D((2, 2)),
        
        layers.Flatten(),
        layers.Dense(256, activation='relu'),
        layers.Dense(num_classes, activation='softmax')
    ])
    return model

# Modelo CON Batch Normalization
def build_model_with_bn(input_shape, num_classes):
    model = keras.Sequential([
        layers.Conv2D(32, (3, 3), padding='same', input_shape=input_shape),
        layers.BatchNormalization(),
        layers.ReLU(),
        layers.Conv2D(32, (3, 3), padding='same'),
        layers.BatchNormalization(),
        layers.ReLU(),
        layers.MaxPooling2D((2, 2)),
        
        layers.Conv2D(64, (3, 3), padding='same'),
        layers.BatchNormalization(),
        layers.ReLU(),
        layers.Conv2D(64, (3, 3), padding='same'),
        layers.BatchNormalization(),
        layers.ReLU(),
        layers.MaxPooling2D((2, 2)),
        
        layers.Flatten(),
        layers.Dense(256),
        layers.BatchNormalization(),
        layers.ReLU(),
        layers.Dense(num_classes, activation='softmax')
    ])
    return model

# Crear ambos modelos
model_no_bn = build_model_no_bn(x_train.shape[1:], num_classes)
model_with_bn = build_model_with_bn(x_train.shape[1:], num_classes)

# Compilar ambos
model_no_bn.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model_with_bn.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# EarlyStopping
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

# Entrenar ambos modelos para comparar
print("\nEntrenando modelo SIN Batch Normalization")
history_no_bn = model_no_bn.fit(
    x_train, y_train_cat,
    batch_size=64,
    epochs=100,
    validation_data=(x_test, y_test_cat),
    callbacks=[early_stop]
)

print("\nEntrenando modelo CON Batch Normalization")
history_with_bn = model_with_bn.fit(
    x_train, y_train_cat,
    batch_size=64,
    epochs=100,
    validation_data=(x_test, y_test_cat),
    callbacks=[early_stop]
)

# Mostrar precisión final
print(f"\nPrecisión sin BN: {history_no_bn.history['val_accuracy'][-1]:.4f}")
print(f"Precisión con BN: {history_with_bn.history['val_accuracy'][-1]:.4f}")

## 2. Data Augmentation

- **Data Augmentation** genera nuevas muestras a partir de datos reales aplicando transformaciones.
- Se usa en **imágenes, texto, audio y otros tipos de datos**.
- Aumenta la diversidad del dataset sin recolectar más datos.
- Ayuda a que el modelo sea **más robusto** y reduzca el sobreajuste.
- Ejemplos clásicos en imágenes: rotaciones, flips, desplazamientos, zoom.

### 2.1 Cutout

- **Cutout** es una técnica de Data Augmentation para imágenes que **oculta aleatoriamente un parche** dentro de cada imagen.
- Obliga al modelo a **no depender solo de una parte específica**, aprendiendo a usar el contexto completo.
- Simula oclusiones reales (por ejemplo, un objeto parcialmente tapado).
- Reduce el riesgo de sobreajuste y mejora la generalización.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def apply_cutout(image, patch_size=16):
    # Aplica Cutout a una sola imagen.
    image_h, image_w, _ = image.shape

    center_x = np.random.randint(0, image_w)
    center_y = np.random.randint(0, image_h)

    x1 = np.clip(center_x - patch_size // 2, 0, image_w)
    x2 = np.clip(center_x + patch_size // 2, 0, image_w)
    y1 = np.clip(center_y - patch_size // 2, 0, image_h)
    y2 = np.clip(center_y + patch_size // 2, 0, image_h)

    cutout_image = image.copy()
    cutout_image[y1:y2, x1:x2, :] = 0
    return cutout_image

# Tomar una imagen de CIFAR-10
sample_image = x_train[0]

plt.figure(figsize=(8, 4))

plt.subplot(1, 2, 1)
plt.imshow(sample_image)
plt.title("Original")
plt.axis('off')

plt.subplot(1, 2, 2)
plt.imshow(apply_cutout(sample_image, patch_size=16))
plt.title("Con Cutout")
plt.axis('off')

plt.show()


In [ ]:
import tensorflow as tf
import numpy as np

# Cargar CIFAR-10
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()
num_classes = 10

# Preprocesar etiquetas
y_train_cat = tf.keras.utils.to_categorical(y_train, num_classes)
y_test_cat = tf.keras.utils.to_categorical(y_test, num_classes)

# Función Cutout
def cutout(image, patch_size=16):
    h, w, c = image.shape
    y = tf.random.uniform([], 0, h, dtype=tf.int32)
    x = tf.random.uniform([], 0, w, dtype=tf.int32)

    y1 = tf.clip_by_value(y - patch_size // 2, 0, h)
    y2 = tf.clip_by_value(y + patch_size // 2, 0, h)
    x1 = tf.clip_by_value(x - patch_size // 2, 0, w)
    x2 = tf.clip_by_value(x + patch_size // 2, 0, w)

    mask = tf.ones((y2 - y1, x2 - x1, c), dtype=image.dtype)
    mask = tf.pad(mask, [[y1, h - y2], [x1, w - x2], [0, 0]], constant_values=0)
    return image * (1 - mask)

# Pipeline tf.data.Dataset
def preprocess(image, label):
    image = tf.cast(image, tf.float32) / 255.0
    image = cutout(image, patch_size=16)
    return image, label

batch_size = 64

train_ds = (
    tf.data.Dataset.from_tensor_slices((x_train, y_train_cat))
    .shuffle(5000)
    .map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(batch_size)
    .prefetch(tf.data.AUTOTUNE)
)

val_ds = (
    tf.data.Dataset.from_tensor_slices((x_test, y_test_cat))
    .map(lambda x, y: (tf.cast(x, tf.float32) / 255.0, y))
    .batch(batch_size)
    .prefetch(tf.data.AUTOTUNE)
)

# Modelo simple
model = tf.keras.Sequential([
    tf.keras.layers.Conv2D(32, 3, activation='relu', input_shape=(32, 32, 3)),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Conv2D(64, 3, activation='relu'),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dense(num_classes, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# EarlyStopping
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

# Entrenar
history = model.fit(
    train_ds,
    epochs=100,
    validation_data=val_ds,
    callbacks=[early_stop]
)

print(f"Precisión final con Cutout: {history.history['val_accuracy'][-1]:.4f}")

### 2.2 Mixup

- **Mixup** es una técnica de Data Augmentation que **combina dos imágenes y sus etiquetas** de forma lineal.
- Crea ejemplos “intermedios” que suavizan las fronteras de decisión.
- Obliga al modelo a no estar excesivamente seguro → mejor generalización y robustez.
- Muy útil para datasets pequeños o ruidosos.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

def apply_mixup(image1, image2, lam=0.5):
    # Combina dos imágenes usando Mixup.
    return lam * image1 + (1 - lam) * image2


(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()
num_classes = 10

# Tomar dos imágenes de CIFAR-10
img1 = x_train[0]
img2 = x_train[1]

# Coeficiente de mezcla (puedes probar otros)
lam = 0.6

# Aplicar Mixup
mixed_image = apply_mixup(img1, img2, lam)

plt.figure(figsize=(12, 4))

plt.subplot(1, 3, 1)
plt.imshow(img1)
plt.title("Imagen 1")
plt.axis('off')

plt.subplot(1, 3, 2)
plt.imshow(img2)
plt.title("Imagen 2")
plt.axis('off')

plt.subplot(1, 3, 3)
plt.imshow(mixed_image.astype(np.uint8))
plt.title(f"Mixup (λ = {lam:.1f})")
plt.axis('off')

plt.show()


In [ ]:
import tensorflow as tf
import numpy as np

# Cargar CIFAR-10
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()
num_classes = 10

# Preprocesar etiquetas
y_train_cat = tf.keras.utils.to_categorical(y_train, num_classes)
y_test_cat = tf.keras.utils.to_categorical(y_test, num_classes)

# Función Mixup
def mixup(batch_x, batch_y, alpha=0.2):
    """Aplica Mixup a un batch."""
    batch_size = tf.shape(batch_x)[0]
    lam = np.random.beta(alpha, alpha)
    index = tf.random.shuffle(tf.range(batch_size))

    mixed_x = lam * batch_x + (1 - lam) * tf.gather(batch_x, index)
    mixed_y = lam * batch_y + (1 - lam) * tf.gather(batch_y, index)
    return mixed_x, mixed_y

# Pipeline tf.data.Dataset
def preprocess(image, label):
    image = tf.cast(image, tf.float32) / 255.0
    return image, label

batch_size = 64

# Crear dataset base
train_ds = (
    tf.data.Dataset.from_tensor_slices((x_train, y_train_cat))
    .shuffle(5000)
    .map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(batch_size)
)

# Aplicar Mixup en cada batch
def mixup_map(batch_x, batch_y):
    return mixup(batch_x, batch_y)

train_ds = train_ds.map(mixup_map, num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE)

# Validación sin Mixup
val_ds = (
    tf.data.Dataset.from_tensor_slices((x_test, y_test_cat))
    .map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(batch_size)
    .prefetch(tf.data.AUTOTUNE)
)

# Modelo simple
model = tf.keras.Sequential([
    tf.keras.layers.Conv2D(32, 3, activation='relu', input_shape=(32, 32, 3)),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Conv2D(64, 3, activation='relu'),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dense(num_classes, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# EarlyStopping
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

# Entrenar
history = model.fit(
    train_ds,
    epochs=100,
    validation_data=val_ds,
    callbacks=[early_stop]
)

print(f"Precisión final con Mixup: {history.history['val_accuracy'][-1]:.4f}")

### 2.3 Label Smoothing

- **Label Smoothing** modifica las etiquetas “duras” para que no sean 100% seguras.
- Por ejemplo, una etiqueta original `[0, 1, 0, 0]` (clase 2) con `label_smoothing=0.1` se convierte en:
  - Clase correcta: `1 - ε = 0.9`
  - Resto de clases: la parte restante `ε = 0.1` se reparte **uniformemente** entre todas las clases.
- Si hay **N clases**, cada clase incorrecta recibe `ε / (N - 1)`.
- En un problema de 4 clases: `[0.033, 0.9, 0.033, 0.033]`
- Esto hace que el modelo:
  - Sea **menos confiado**, evitando overfitting.
  - Aprenda fronteras de decisión más suaves.
  - Mejore su calibración para casos ambiguos.



In [ ]:
import tensorflow as tf

# Cargar CIFAR-10
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()
num_classes = 10

# Normalizar y etiquetas
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0
y_train_cat = tf.keras.utils.to_categorical(y_train, num_classes)
y_test_cat = tf.keras.utils.to_categorical(y_test, num_classes)

# Modelo simple
model = tf.keras.Sequential([
    tf.keras.layers.Conv2D(32, 3, activation='relu', input_shape=(32, 32, 3)),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Conv2D(64, 3, activation='relu'),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dense(num_classes, activation='softmax')
])

# Compilar con Label Smoothing
model.compile(
    optimizer='adam',
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=['accuracy']
)

# EarlyStopping
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

# Entrenar
history = model.fit(
    x_train, y_train_cat,
    epochs=100,
    batch_size=64,
    validation_data=(x_test, y_test_cat),
    callbacks=[early_stop]
)

print(f"Precisión final con Label Smoothing: {history.history['val_accuracy'][-1]:.4f}")


- Con estas técnicas (Batch Normalization, Data Augmentation avanzada - Cutout y Mixup y Label Smoothing) podemos entrenar redes profundas de forma más estable y robusta.
- Cada una resuelve un problema diferente: BatchNorm estabiliza activaciones internas; Cutout y Mixup hacen al modelo más resistente y reducen sobreajuste; Label Smoothing controla la sobreconfianza y mejora la calibración.
- Todas se pueden combinar fácilmente con arquitecturas avanzadas como ResNet, Inception o EfficientNet.
- Probar y ajustar estas estrategias es clave para lograr modelos confiables y con mejor capacidad de generalización en escenarios reales.
